# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [3]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx datasets python-dotenv requests==2.32.4


In [4]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

# Colab Secrets is preferred. For a private runtime, an uploaded /content/.env
# or a .env next to the notebook is also supported. Values are never printed.
from dotenv import load_dotenv

ENV_CANDIDATES = [Path("/content/.env"), Path.cwd() / ".env"]
for env_path in ENV_CANDIDATES:
    if env_path.is_file():
        load_dotenv(env_path, override=False)
        print(f"Loaded environment variables from {env_path}")
        break

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

LLM_PROVIDER = get_secret("LLM_PROVIDER", "openai").lower()
LLM_MODEL = get_secret("LLM_MODEL", "gpt-4o-mini")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "/content/hackernoon_subset.csv"
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These row ids are the verified evidence for the five selected Golden queries.
# They are prioritized before deterministic sampling/chunk caps.
GOLDEN_EVIDENCE_ROW_IDS = {33, 648, 935, 1187, 1229, 1435}
REPO_ROOT = Path.cwd()
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# These row ids are the verified evidence for the five selected Golden queries.
# They are prioritized before deterministic sampling/chunk caps.
GOLDEN_EVIDENCE_ROW_IDS = {33, 648, 935, 1187, 1229, 1435}
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40


# CONFIG_VALIDATION_V2
required_secrets = {
    "NEO4J_URI": NEO4J_URI,
    "NEO4J_PASSWORD": NEO4J_PASSWORD,
    "HF_TOKEN": HF_TOKEN,
    "LLM_MODEL": LLM_MODEL,
    "JUDGE_MODEL": JUDGE_MODEL,
}
missing_secrets = [name for name, value in required_secrets.items() if not value]
if LLM_PROVIDER == "openai" or JUDGE_PROVIDER == "openai":
    if not OPENAI_API_KEY:
        missing_secrets.append("OPENAI_API_KEY")
if LLM_PROVIDER == "groq":
    if not GROQ_API_KEY:
        missing_secrets.append("GROQ_API_KEY")
    if not GROQ_MODEL:
        missing_secrets.append("GROQ_MODEL")
if missing_secrets:
    raise RuntimeError(
        "Missing secrets: " + ", ".join(sorted(set(missing_secrets)))
        + ". Add them to Colab Secrets or upload a private /content/.env."
    )


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [5]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 5000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

Đang ghi dữ liệu vào: /content/hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]


[DỪNG] Đã đạt giới hạn số dòng: 5,000 dòng (Dung lượng: 2.92 MB)
✅ Hoàn thành: /content/hackernoon_subset.csv
   Rows: 5,000
   Size: 2.92 MB


In [6]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

# connect_neo4j()
# setup_graph_schema()

# PIPELINE_RUN_M1_NEO4J_V2
connect_neo4j()
setup_graph_schema()


✅ Neo4j connected.
✅ Schema ready.


In [7]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["source_row_id"] = np.arange(len(raw), dtype=int)
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        priority = df[df.source_row_id.isin(GOLDEN_EVIDENCE_ROW_IDS)]
        remaining = df[~df.source_row_id.isin(GOLDEN_EVIDENCE_ROW_IDS)]
        slots = max(0, LAB_MAX_ARTICLES - len(priority))
        remaining = remaining.sample(min(slots, len(remaining)), random_state=SEED)
        df = pd.concat([priority, remaining], ignore_index=True)
        df = df.sort_values("source_row_id").reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    priority = news_df[news_df.source_row_id.isin(GOLDEN_EVIDENCE_ROW_IDS)]
    remaining = news_df[~news_df.source_row_id.isin(GOLDEN_EVIDENCE_ROW_IDS)]
    ordered_news = pd.concat([priority, remaining], ignore_index=True)
    for r in tqdm(ordered_news.itertuples(index=False), total=len(ordered_news), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "source_row_id": int(r.source_row_id),
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())


Exact dedup: 2,675 -> 2,105


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

,chunk_id,article_id,source_row_id,title,published_date,text
0,4f1346392056a403277d::c0000,4f1346392056a403277d,33,Aeris to Acquire IoT Business from Ericsson,2022-12-07,Aeris Communications and Ericsson are joining together to create a leader in the fast-growing IoT industry Ericsson'...
1,a5a8b0ece135c638d4f0::c0000,a5a8b0ece135c638d4f0,648,ServiceNow NVIDIA and Accenture Partner to Accelerate Generative AI Adoption for Enterprises,2023-07-27,ServiceNow NVIDIA and Accenture today announced the launch of AI Lighthouse a first-of-its-kind program designed
2,e0ebf26fe86416644315::c0000,e0ebf26fe86416644315,935,A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular IoT,2023-01-18,After signing a recent agreement Aeris has acquired Ericsson''s IoT Accelerator and Connected Vehicle Cloud business...
3,a2fe2cc636f39a18f032::c0000,a2fe2cc636f39a18f032,1187,Options Announces Achievement of Microsoft Infrastructure Solutions Partner Status,2023-05-10,Options Technology the leading provider of cloud-enabled managed services to the global capital markets today announ...
4,f73febf774301e802436::c0000,f73febf774301e802436,1229,Options Named Microsoft Solutions Partner for Security,2023-06-08,Options Technology the leading provider of cloud-enabled managed services to the global capital markets today announ...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [8]:
#@title 1.6 — OpenAI-only LLM wrapper

from openai import OpenAI

OPENAI_API_KEY = get_secret(
    "OPENAI_API_KEY",
    ""
)

LLM_PROVIDER = "openai"
LLM_MODEL = get_secret(
    "LLM_MODEL",
    "gpt-4o-mini"
)

if not OPENAI_API_KEY:
    raise RuntimeError(
        "Missing OPENAI_API_KEY."
    )

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)


def parse_json_object(text):
    text = str(text).strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    start = text.find("{")
    end = text.rfind("}")

    if start < 0 or end <= start:
        raise ValueError(
            "No JSON object found."
        )

    return json.loads(
        text[start:end + 1]
    )


def llm_chat(
    messages,
    model=None,
    json_mode=False,
    max_retries=4,
    provider=None
):
    model = model or LLM_MODEL

    last_error = None

    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0
            }

            if json_mode:
                kwargs["response_format"] = {
                    "type": "json_object"
                }

            response = (
                openai_client
                .chat
                .completions
                .create(**kwargs)
            )

            usage = {}

            if response.usage is not None:
                usage = {
                    "prompt_tokens": (
                        response.usage.prompt_tokens
                    ),
                    "completion_tokens": (
                        response.usage.completion_tokens
                    ),
                    "total_tokens": (
                        response.usage.total_tokens
                    )
                }

            return (
                response.choices[0].message.content,
                usage
            )

        except Exception as error:
            last_error = error

            if attempt < max_retries - 1:
                wait_time = min(
                    20,
                    2 ** attempt + random.random()
                )
                time.sleep(wait_time)

    raise RuntimeError(
        f"OpenAI request failed: {last_error}"
    )


def llm_json(
    system,
    user,
    model=None,
    provider=None
):
    text, usage = llm_chat(
        messages=[
            {
                "role": "system",
                "content": system
            },
            {
                "role": "user",
                "content": user
            }
        ],
        model=model or LLM_MODEL,
        json_mode=True
    )

    return parse_json_object(text), usage


# Compatibility aliases:
# các cell cũ gọi groq_chat/groq_json
# nhưng thực tế vẫn dùng OpenAI
def groq_chat(
    messages,
    model=None,
    json_mode=False,
    max_retries=4
):
    return llm_chat(
        messages=messages,
        model=model or LLM_MODEL,
        json_mode=json_mode,
        max_retries=max_retries,
        provider="openai"
    )


def groq_json(
    system,
    user,
    model=None
):
    return llm_json(
        system=system,
        user=user,
        model=model or LLM_MODEL,
        provider="openai"
    )


print("✅ OpenAI-only wrapper ready.")
print(f"Pipeline model: {LLM_MODEL}")

✅ OpenAI-only wrapper ready.
Pipeline model: gpt-4o-mini


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [9]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = llm_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    if chunks_subset.empty:
        return pd.DataFrame(columns=["chunk_id", "resolved_text", "unresolved_mentions"])
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")


Coref:   0%|          | 0/80 [00:00<?, ?it/s]

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [10]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return llm_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    triple_columns = [
        "source_raw", "source_type", "relation", "target_raw", "target_type",
        "source_chunk_id", "published_date", "evidence", "confidence",
    ]
    error_columns = ["start", "error"]
    return pd.DataFrame(triples, columns=triple_columns), pd.DataFrame(errors, columns=error_columns)

# raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
# display(raw_triples_df.head())


# PIPELINE_RUN_M2_EXTRACTION_V2
raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
if raw_triples_df.empty:
    display(extraction_errors_df.head(20))
    raise RuntimeError(
        "NER/RE produced zero triples. Inspect extraction_errors_df; do not continue with an empty graph."
    )
print({"triples": len(raw_triples_df), "extraction_errors": len(extraction_errors_df)})
display(raw_triples_df.head())


NER+RE:   0%|          | 0/100 [00:00<?, ?it/s]

{'triples': 170, 'extraction_errors': 0}


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Aeris,Company,ACQUIRED,Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses,Technology,e0ebf26fe86416644315::c0000,2023-01-18,Aeris has acquired Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses,1.0
1,Aeris Communications,Company,PARTNERED_WITH,Ericsson,Company,4f1346392056a403277d::c0000,2022-12-07,Aeris Communications and Ericsson are joining together,1.0
2,ServiceNow,Company,PARTNERED_WITH,NVIDIA,Company,a5a8b0ece135c638d4f0::c0000,2023-07-27,"ServiceNow, NVIDIA, and Accenture today announced the launch of AI Lighthouse",1.0
3,ServiceNow,Company,PARTNERED_WITH,Accenture,Company,a5a8b0ece135c638d4f0::c0000,2023-07-27,"ServiceNow, NVIDIA, and Accenture today announced the launch of AI Lighthouse",1.0
4,NVIDIA,Company,PARTNERED_WITH,Accenture,Company,a5a8b0ece135c638d4f0::c0000,2023-07-27,"ServiceNow, NVIDIA, and Accenture today announced the launch of AI Lighthouse",1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [11]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    audit_columns = ["type", "left", "right", "similarity", "decision"]
    return mapping, pd.DataFrame(audit, columns=audit_columns)

def canonicalize_triples(raw_df, mapping):
    required = {"source_raw", "source_type", "target_raw", "target_type"}
    missing = required - set(raw_df.columns)
    if raw_df.empty:
        raise ValueError("raw_triples_df is empty; fix NER/RE extraction first.")
    if missing:
        raise ValueError(f"raw_triples_df missing columns: {sorted(missing)}")
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

# entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
# triples_df = canonicalize_triples(raw_triples_df, entity_map)
# display(entity_resolution_audit_df.head(20))


# PIPELINE_RUN_M2_RESOLUTION_V2
entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
triples_df = canonicalize_triples(raw_triples_df, entity_map)
print(entity_resolution_audit_df.decision.value_counts(dropna=False))
display(entity_resolution_audit_df.sort_values("similarity", ascending=False).head(20))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

decision
MERGE_VECTOR    3
Name: count, dtype: int64


,type,left,right,similarity,decision
1,Company,L&T Technology Services Limited,L&T Technology Services,0.925773,MERGE_VECTOR
0,Company,Fidelity National Information Services,Fidelity National Information Services Inc.,0.924524,MERGE_VECTOR
2,Technology,Microsoft Solutions Partner designation for Infrastructure,Microsoft Solutions Partner designation,0.921302,MERGE_VECTOR


In [12]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

# nodes_df = build_nodes(triples_df)
# bulk_insert_nodes(nodes_df)
# bulk_insert_edges(triples_df)

# PIPELINE_RUN_M2_INGEST_V2
nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)
print({"ingested_nodes": len(nodes_df), "ingested_edges": len(triples_df)})


{'ingested_nodes': 286, 'ingested_edges': 169}


In [ ]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

# graph_counts, top_degree_df = graph_checks()

# PIPELINE_RUN_M2_CHECKS_V2
graph_counts, top_degree_df = graph_checks()


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [13]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

# build_flat_index(chunks_df)

# PIPELINE_RUN_M3_FLAT_V2
build_flat_index(chunks_df)


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [14]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = llm_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

# build_entity_matcher(nodes_df)

# PIPELINE_RUN_M3_MATCHER_V2
build_entity_matcher(nodes_df)


In [15]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [16]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = llm_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=LLM_MODEL,
        provider=LLM_PROVIDER
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out


# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [17]:
#@title 4.1 — Verified 5-query Golden Dataset
GOLDEN_PATH_CANDIDATES = [
    Path("/content/golden_dataset.csv"),
    REPO_ROOT / "data" / "golden_dataset.csv",
]

starter_golden = pd.DataFrame([
    {"id":"G5000-03","group":"factoid","question":"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the later connectivity report?","reference_answer":"More than 100 million IoT devices, 9,000 enterprises, and 190 countries.","reference_evidence":"row 935: A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular IoT"},
    {"id":"G5000-05","group":"multi-hop","question":"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale should be returned?","reference_answer":"Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more than 100 million IoT devices for 9,000 enterprises across 190 countries.","reference_evidence":"rows 33 and 935: Aeris/Ericsson acquisition and later connectivity report"},
    {"id":"G5000-07","group":"cross-doc","question":"How did ServiceNow's July generative-AI expansion differ between its platform features and its ecosystem program with NVIDIA and Accenture?","reference_answer":"ServiceNow expanded the Now Assist feature set with case summarization and text-to-code inside its workflow platform, while separately joining NVIDIA and Accenture to launch AI Lighthouse, an enterprise generative-AI adoption program.","reference_evidence":"rows 1435 and 648: ServiceNow platform expansion and AI Lighthouse announcement"},
    {"id":"G5000-13","group":"factoid","question":"Which three companies launched AI Lighthouse in the selected 5,000-row scope?","reference_answer":"ServiceNow, NVIDIA, and Accenture.","reference_evidence":"row 648: ServiceNow, NVIDIA and Accenture partner announcement"},
    {"id":"G5000-20","group":"cross-doc","question":"How did Options Technology's Microsoft partner designations expand from May to June 2023?","reference_answer":"In May, Options Technology announced Microsoft Solutions Partner status for Infrastructure. In June, it announced Microsoft Solutions Partner status for Security, expanding the documented specialization from infrastructure to security.","reference_evidence":"rows 1187 and 1229: Infrastructure and Security partner announcements"},
])

golden_path = next((p for p in GOLDEN_PATH_CANDIDATES if p.is_file()), None)
golden_df = pd.read_csv(golden_path) if golden_path else starter_golden.copy()

def validate_golden(df, require_answers=True):
    required = {"id", "group", "question", "reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id", "question"]])
        raise ValueError("Fill reference_answer before final evaluation.")
    required_groups = {"factoid", "multi-hop", "cross-doc"}
    if not required_groups.issubset(set(df.group)):
        raise ValueError(f"Golden Dataset must contain {sorted(required_groups)}")
    if len(df) < 5:
        raise ValueError("Golden Dataset must contain at least five queries.")
    print("✅ Golden Dataset valid.")

validate_golden(golden_df, require_answers=True)
display(golden_df)


✅ Golden Dataset valid.


,id,group,question,reference_answer,reference_evidence
0,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935: A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular IoT
1,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,rows 33 and 935: Aeris/Ericsson acquisition and later connectivity report
2,G5000-07,cross-doc,How did ServiceNow's July generative-AI expansion differ between its platform features and its ecosystem program wit...,ServiceNow expanded the Now Assist feature set with case summarization and text-to-code inside its workflow platform...,rows 1435 and 648: ServiceNow platform expansion and AI Lighthouse announcement
3,G5000-13,factoid,"Which three companies launched AI Lighthouse in the selected 5,000-row scope?","ServiceNow, NVIDIA, and Accenture.","row 648: ServiceNow, NVIDIA and Accenture partner announcement"
4,G5000-20,cross-doc,How did Options Technology's Microsoft partner designations expand from May to June 2023?,"In May, Options Technology announced Microsoft Solutions Partner status for Infrastructure. In June, it announced Mi...",rows 1187 and 1229: Infrastructure and Security partner announcements


In [18]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    return llm_json(system, user, model=JUDGE_MODEL, provider=JUDGE_PROVIDER)[0]

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out


In [19]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = "/content/graphrag_eval_checkpoint.csv"

def run_evaluation(golden_df, checkpoint=CHECKPOINT):
    checkpoint = Path(checkpoint)
    rows = []
    completed = set()
    if checkpoint.is_file():
        previous = pd.read_csv(checkpoint)
        previous = previous[previous.id.astype(str).isin(golden_df.id.astype(str))]
        rows = previous.to_dict("records")
        completed = set(previous["id"].astype(str))
        print(f"Resuming evaluation: {len(completed)} completed queries.")
    pending = golden_df[~golden_df.id.astype(str).isin(completed)]
    for q in tqdm(pending.itertuples(index=False), total=len(pending), desc="Evaluation"):
        flat = answer_flat_rag(q.question)
        graph = answer_graph_rag(q.question)

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(checkpoint, index=False)
    result = pd.DataFrame(rows)
    order = {str(qid): i for i, qid in enumerate(golden_df.id)}
    return result.assign(_order=result.id.astype(str).map(order)).sort_values("_order").drop(columns="_order").reset_index(drop=True)

# validate_golden(golden_df, require_answers=True)
# eval_results_df = run_evaluation(golden_df)
# display(eval_results_df)


# PIPELINE_RUN_M4_EVAL_V2
validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)


✅ Golden Dataset valid.


Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.","The later connectivity report cited more than 100 million IoT devices, 9,000 enterprises, and 190 countries [chunk_i...","The later connectivity report cited more than 100 million IoT devices, 9,000 enterprises, and 190 countries [chunk_i...",5,5,5,5,5,5,1.080608,1.019796,575,565,"The candidate accurately cites the numbers of IoT devices, enterprises, and countries as stated in the reference. It...","The candidate accurately cites the numbers of IoT devices, enterprises, and countries as stated in the reference ans...",0
1,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,The path from Ericsson to the acquirer is as follows: Ericsson's IoT Accelerator and Connected Vehicle Cloud busines...,The path from Ericsson to the acquirer is as follows: Aeris Communications acquired Ericsson's IoT Accelerator and C...,5,5,5,5,5,5,1.570554,1.318673,641,533,"The candidate provides a complete and accurate path from Ericsson to Aeris, detailing the acquisition of Ericsson's ...","The candidate accurately follows the path from Ericsson to Aeris Communications, correctly identifying the acquisiti...",0
2,G5000-07,cross-doc,How did ServiceNow's July generative-AI expansion differ between its platform features and its ecosystem program wit...,ServiceNow expanded the Now Assist feature set with case summarization and text-to-code inside its workflow platform...,ServiceNow's July generative-AI expansion included enhancements to its platform features and the launch of an ecosys...,ServiceNow's July generative-AI expansion included two distinct components: enhancements to its platform features an...,4,5,4,5,4,5,2.385421,3.002400,706,1640,"The candidate provides a clear distinction between the platform features and the ecosystem program, mentioning speci...","The candidate provides a detailed and accurate comparison of ServiceNow's July generative-AI expansion, clearly dist...",0
3,G5000-13,factoid,"Which three companies launched AI Lighthouse in the selected 5,000-row scope?","ServiceNow, NVIDIA, and Accenture.","The three companies that launched AI Lighthouse are ServiceNow, NVIDIA, and Accenture [chunk_id=a5a8b0ece135c638d4f0...","The three companies that launched AI Lighthouse are ServiceNow, NVIDIA, and Accenture [chunk_id=a5a8b0ece135c638d4f0...",5,5,5,5,5,5,1.081853,0.916693,528,380,"The candidate accurately identifies the three companies that launched AI Lighthouse as ServiceNow, NVIDIA, and Accen...","The candidate accurately identifies the three companies that launched AI Lighthouse as ServiceNow, NVIDIA, and Accen...",0
4,G5000-20,cross-doc,How did Options Technology's Microsoft partner designations expand from May to June 2023?,"In May, Options Technology announced Microsoft Solutions Partner status for Infrastructure. In June, it announced Mi...",Options Technology expanded its Microsoft partner designations from May to June 2023 by achieving the Microsoft Solu...,Options Technology expanded its Microsoft partner designations from May to June 2023 by achieving the Microsoft Solu...,5,4,5,5,5,4,1.270420,1.828988,593,1116,The candidate accurately captures the expansion of Options Technology's Microsoft partner designations by stating th...,"The candidate provides a clear timeline of Options Techn

In [20]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

# comparison_df = comparison_table(eval_results_df)
# display(comparison_df)
# eval_results_df.to_csv("/content/graphrag_eval_results.csv", index=False)
# comparison_df.to_csv("/content/graphrag_vs_flatrag_summary.csv", index=False)

# PIPELINE_RUN_M4_EXPORT_V2
comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_path = OUTPUT_DIR / "graphrag_eval_results.csv"
summary_path = OUTPUT_DIR / "graphrag_vs_flatrag_summary.csv"
eval_results_df.to_csv(eval_path, index=False)
comparison_df.to_csv(summary_path, index=False)
print(f"Saved: {eval_path}")
print(f"Saved: {summary_path}")


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,4.500,4.500,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,4.500,5.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,4.500,4.500,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),1.828,2.416,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,649.500,1378.000,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,5.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.081,0.968,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,551.500,472.500,GraphRAG không đắt hơn trong sample này.


Saved: /content/outputs/graphrag_eval_results.csv
Saved: /content/outputs/graphrag_vs_flatrag_summary.csv


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [22]:
#@title 5.1 — Super-node check + entity audit

def test_supernode_policy():
    rows = run_cypher("""
        MATCH (n:Entity)-[r]-()
        WITH n, count(r) AS degree
        ORDER BY degree DESC
        LIMIT 1
        RETURN
            n.id AS id,
            n.name AS name,
            n.entity_type AS type,
            degree
    """)

    if not rows:
        print("⚠️ Graph empty.")
        return None

    top_node = rows[0]

    limit = (
        SUPER_NODE_EDGE_CAP
        if top_node["degree"] > SUPER_NODE_DEGREE
        else 1000
    )

    edges = recent_edges(
        top_node["id"],
        limit
    )

    print(
        "Top node:",
        top_node
    )

    print(
        "Fetched edges:",
        len(edges)
    )

    if top_node["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= SUPER_NODE_EDGE_CAP
        print(
            "✅ Super-node cap OK: "
            f"{len(edges)} <= {SUPER_NODE_EDGE_CAP}"
        )
    else:
        print(
            "ℹ️ Không có node nào vượt "
            f"degree {SUPER_NODE_DEGREE}."
        )

    return top_node


def show_resolution_audit(audit_df):
    if audit_df is None or audit_df.empty:
        print("⚠️ No entity-resolution audit rows.")
        return

    print(
        f"Audit rows: {len(audit_df):,}"
    )

    display(
        audit_df
        .sort_values(
            "similarity",
            ascending=False
        )
        .head(30)
    )

    rejected = audit_df[
        audit_df["decision"] == "REJECT_GUARD"
    ]

    print(
        f"REJECT_GUARD rows: {len(rejected):,}"
    )

    if not rejected.empty:
        display(
            rejected
            .sort_values(
                "similarity",
                ascending=False
            )
            .head(20)
        )


# =========================
# Tạo graph integrity summary
# =========================

invalid_provenance = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL
       OR r.published_date IS NULL
    RETURN count(r) AS count
""")[0]["count"]

node_count = run_cypher("""
    MATCH (n:Entity)
    RETURN count(n) AS count
""")[0]["count"]

edge_count = run_cypher("""
    MATCH ()-[r]->()
    RETURN count(r) AS count
""")[0]["count"]

graph_counts = {
    "nodes": node_count,
    "edges": edge_count,
    "invalid_provenance_edges": invalid_provenance
}

print("Graph integrity:")
print(graph_counts)

assert invalid_provenance == 0, (
    "Có edge thiếu source_chunk_id "
    "hoặc published_date."
)


# =========================
# Tạo top degree entities
# =========================

top_degree_df = pd.DataFrame(
    run_cypher("""
        MATCH (n:Entity)
        OPTIONAL MATCH (n)-[r]-()
        WITH
            n,
            count(r) AS degree
        RETURN
            n.id AS id,
            n.name AS name,
            n.entity_type AS type,
            degree
        ORDER BY degree DESC
        LIMIT 15
    """)
)

display(top_degree_df)


# =========================
# Chạy các failure-mode checks
# =========================

test_supernode_policy()

show_resolution_audit(
    entity_resolution_audit_df
)


# =========================
# Lưu các file audit
# =========================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

entity_resolution_audit_df.to_csv(
    OUTPUT_DIR / "entity_resolution_audit.csv",
    index=False
)

top_degree_df.to_csv(
    OUTPUT_DIR / "top_degree_entities.csv",
    index=False
)

pd.DataFrame(
    [graph_counts]
).to_csv(
    OUTPUT_DIR / "graph_integrity_summary.csv",
    index=False
)

print(
    "✅ Required implementation, "
    "evaluation, and failure-mode checks completed."
)

Graph integrity:
{'nodes': 680, 'edges': 448, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,fcc6c88e2bd1527cb4cf67f1,The Fourth In America,Technology,9
1,ed73b9eff3f5ff1eea6154b7,Xi Jinping,Person,7
2,7b219357277193c25d37d788,Railergy,Company,6
3,e6c3db5c28c9a15bfafd2e04,Apple,Company,5
4,b0f073fc691ac6ea5bfa2c41,Activision Blizzard,Company,5
5,8799d8067b0eb4a83af1e4e7,adidas,Company,5
6,4f5c599d80e3f459819f8d8f,AI,Technology,5
7,c7c95b849d2fd4f07ae49bfd,Byju's,Company,4
8,b823fb02e58a12bdc4c38934,NVIDIA,Company,4
9,24efe6cedc09a12ec8c3eff2,Synopsys,Company,4


Top node: {'id': 'fcc6c88e2bd1527cb4cf67f1', 'name': 'The Fourth In America', 'type': 'Technology', 'degree': 9}
Fetched edges: 9
ℹ️ Không có node nào vượt degree 100.
Audit rows: 3


,type,left,right,similarity,decision
1,Company,L&T Technology Services Limited,L&T Technology Services,0.925773,MERGE_VECTOR
0,Company,Fidelity National Information Services,Fidelity National Information Services Inc.,0.924524,MERGE_VECTOR
2,Technology,Microsoft Solutions Partner designation for Infrastructure,Microsoft Solutions Partner designation,0.921302,MERGE_VECTOR


REJECT_GUARD rows: 0
✅ Required implementation, evaluation, and failure-mode checks completed.


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [23]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [24]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau